In [12]:
pkl.keys()
# pkl["meta"]
# pkl['sweep']
pkl["results"]

[{'params': {'K_p': 0.015,
   'K_i': array([1800., 1800., 1800., 1800., 1800., 1800., 1800., 1800., 1800.,
          1800., 1800., 1800., 1800., 1800., 1800., 1800., 1800., 1800.,
          1800., 1800., 1800., 1800., 1800., 1800., 1800., 1800., 1800.,
          1800., 1800., 1800., 1800.]),
   'R_c': 1000.0,
   'amp': 6.25,
   'K_c': array([8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09,
          8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09,
          8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09,
          8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09, 8.e+09])},
  'data': {'freq': array([    0.        ,    99.93337775,   199.8667555 ,   299.80013324,
            399.73351099,   499.66688874,   599.60026649,   699.53364424,
            799.46702199,   899.40039973,   999.33377748,  1099.26715523,
           1199.20053298,  1299.13391073,  1399.06728847,  1499.00066622,
           1598.93404397,  1698.86742172,  1798.80079947,  1898.7341772

In [13]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path

# =========================================================
# USER: LOAD RUN
# =========================================================
run_dir = Path("sim_dat/sweep_3")   # <-- change if needed
pkl_path = run_dir / "results.pkl"

with open(pkl_path, "rb") as f:
	pkl = pickle.load(f)

results = pkl["results"]
sweep_keys = pkl["sweep"]["keys"]

print("Sweep keys:", sweep_keys)
print(f"Loaded {len(results)} results")

# =========================================================
# HELPER: filter results by fixed parameters
# =========================================================
def filter_results(results, fixed_params):
	"""
	fixed_params: dict of {param_name: value}
	Returns list of matching result dicts
	"""
	out = []
	for r in results:
		ok = True
		for k, v in fixed_params.items():
			if k not in r["params"]:
				ok = False
				break
			if r["params"][k] != v:
				ok = False
				break
		if ok:
			out.append(r)
	return out

# =========================================================
# HELPER: group results by a parameter
# =========================================================
def group_by(results, key):
	groups = defaultdict(list)
	for r in results:
		groups[r["params"][key]].append(r)
	return groups

# =========================================================
# USER: WHAT TO PLOT
# =========================================================
#
# Choose:
#   - x-axis: always frequency
#   - y-data: FRF
#   - group curves by ONE parameter
#   - optionally fix other parameters
#

GROUP_BY = "amp"          # e.g. "amp", "kc", "K_p"
FIX_PARAMS = {
	# "kc": 2e10,         # uncomment to fix kc
	# "K_p": 0.015,       # uncomment to fix Kp
}

# =========================================================
# FILTER + GROUP
# =========================================================
filtered = filter_results(results, FIX_PARAMS)

if len(filtered) == 0:
	raise RuntimeError("No results match FIX_PARAMS")

groups = group_by(filtered, GROUP_BY)

print(f"Plotting {len(filtered)} curves")
print(f"Grouped by '{GROUP_BY}' → {len(groups)} groups")

# =========================================================
# PLOT
# =========================================================
plt.figure(figsize=(8, 5))

colors = plt.cm.viridis(np.linspace(0, 1, len(groups)))

for (val, group), col in zip(sorted(groups.items()), colors):
	for r in group:
		freq = r["data"]["freq"]
		FRF  = r["data"]["FRF"]
		plt.semilogy(freq, FRF, color=col, alpha=0.8)

	plt.plot([], [], color=col, label=f"{GROUP_BY} = {val}")

plt.xlabel("Frequency [Hz]")
plt.ylabel("FRF")
plt.grid(True, which="both")
plt.legend()
plt.tight_layout()
plt.xlim([1000, 3000])
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'sim_dat\\sweep_3\\results.pkl'